In [19]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
import re
import joblib
import string

fake=pd.read_csv('Fake.csv')
true=pd.read_csv('True.csv')

In [20]:
fake.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [21]:
true.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [22]:
fake['class']=0
true['class']=1

In [23]:
data=pd.concat([fake,true],axis=0)

In [24]:
data.sample(10)

,title,text,subject,date,class
4780,Detroit Reverend BLASTS Trump’s ‘Scam’ He Jus...,Donald Trump s visit to a historically black c...,News,"September 3, 2016",0
23455,"BOILER ROOM – Examination, Exclamation, Excita...",Tune in to the Alternate Current Radio Network...,Middle-east,"February 3, 2016",0
7256,Bundy Militia Fans Continue Threats To Kill O...,More than two months after Bundy Militia idiot...,News,"March 27, 2016",0
15647,NO TRANSPARENCY ON TRADE BILL FROM THE MOST TR...,WHEN YOU AGREE WITH LIZ WARREN YOU KNOW SOMETH...,politics,"May 22, 2015",0
2472,Trump Just Got DESTROYED By J.K. Rowling For ...,Donald Trump topped off his disastrous press c...,News,"February 18, 2017",0
15440,(VIDEO) HUCKSTER AL SHARPTON: CLIMATE CHANGE I...,The global warming hucksterism in the Obama ad...,politics,"Jul 21, 2015",0
7204,WATCH: Anderson Cooper DESTROYS Ted Cruz Over...,Cruz has recently gone on the attack against D...,News,"March 30, 2016",0
17397,OBAMA KNEW DRONE KILLED U.S. HOSTAGE…Never Tol...,Why couldn t the parents be notified of his de...,Government News,"Apr 25, 2015",0
11078,White House declines comment on Johnson Contro...,WASHINGTON (Reuters) - The White House on Wedn...,politicsNews,"January 27, 2016",1
11448,Turkey wants to bring wounded from Syria's Gho...,ISTANBUL (Reuters) - Turkey is working with Ru...,worldnews,"December 24, 2017",1


In [25]:
data=data.drop(['title','subject','date'],axis=1)

In [26]:
data.reset_index(inplace=True)

In [27]:
data.drop(['index'],axis=1,inplace=True)

In [28]:
data.sample(5)

,text,class
31492,NEW YORK (Reuters) - Democratic presidential n...,1
16880,Environmental Protection Agency (EPA) enforcer...,0
41860,TOKYO (Reuters) - Japanese Prime Minister Shin...,1
11492,https://www.youtube.com/watch?v=n9tfNMQpYWU,0
28474,NEW YORK (Reuters) - The Trump administration ...,1


In [29]:
def clean_text(text):
  text=text.lower()
  text=re.sub(r'\[.*?\]',"",text)
  text=re.sub(r"\\w"," ",text)
  text=re.sub(r"https?:://\S+|www\.\S+","",text)
  text=re.sub(r"<.*?>+","",text)
  text=re.sub(r"[%s]" % re.escape(string.punctuation),"",text)
  text=re.sub(r"\n","",text)
  text=re.sub(r"\w*\d\w*","",text)
  return text

In [30]:
data["text"]=data["text"].apply(clean_text)

In [31]:
x=data["text"]
y=data["class"]
xtrain,xtest,ytrain,ytest=train_test_split(x,y,test_size=0.25,random_state=42)

In [32]:
vectorizer=TfidfVectorizer()
xv_train=vectorizer.fit_transform(xtrain)
xv_test=vectorizer.transform(xtest)

In [33]:
lr=LogisticRegression()
lr.fit(xv_train,ytrain)

LogisticRegression()

In [34]:
prediction=lr.predict(xv_test)
lr.score(xv_test,ytest)

0.985924276169265

In [35]:
print(classification_report(ytest,prediction))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      5895
           1       0.98      0.99      0.99      5330

    accuracy                           0.99     11225
   macro avg       0.99      0.99      0.99     11225
weighted avg       0.99      0.99      0.99     11225



In [37]:
joblib.dump(vectorizer,'vectorizer.jb')
joblib.dump(lr,'lr_model.jb')

['lr_model.jb']